# Gradient Descent and Its Variants

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 04.01 |
| Time | ~60 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/04_Optimization/02_gradient_descent_variants.ipynb)

---

## 🎯 Learning Objective

Implement batch, stochastic, and mini-batch gradient descent, and compare their convergence behavior as a function of learning rate.

---

## 📐 Theory

`04.01` classified the terrain; this notebook studies the actual vehicle that crosses it. Every
version of gradient descent shares the same update rule, $\mathbf{w}\leftarrow\mathbf{w}-\eta\nabla
L(\mathbf{w})$ (recall from `02.05` that $-\nabla L$ is the direction of steepest local decrease),
and the three variants below differ only in *how much data* they use to estimate $\nabla L$ before
each step.

### Batch gradient descent

For a loss averaged over $N$ training examples, $L(\mathbf{w})=\frac{1}{N}\sum_{i=1}^N
\ell_i(\mathbf{w})$, **batch gradient descent** computes the exact gradient using every example:
$\nabla L(\mathbf{w}) = \frac{1}{N}\sum_i \nabla \ell_i(\mathbf{w})$. Each step is the truest
possible estimate of the downhill direction, so the trajectory is smooth. The cost: one step
requires touching all $N$ examples, which is infeasible when $N$ is in the millions or billions.

### Stochastic gradient descent (SGD)

**SGD** replaces the exact gradient with a one-example estimate: pick a random index $i$ and
step using $\nabla \ell_i(\mathbf{w})$ alone. This is an unbiased estimator of the true gradient
($\mathbb{E}_i[\nabla \ell_i(\mathbf{w})] = \nabla L(\mathbf{w})$), but a high-variance one — each
step is cheap ($O(1)$ instead of $O(N)$) but noisy, so the path zigzags even while trending
downhill on average.

### Mini-batch gradient descent

**Mini-batch** interpolates between the two: average the gradient over a small random batch of
$B$ examples ($1 < B \ll N$). Averaging $B$ independent noisy estimates reduces the variance of
the gradient estimate by a factor of $B$ (the standard result for averaging i.i.d. samples), so
mini-batch trades some of SGD's noise for a cost of $O(B)$ per step instead of $O(1)$ or $O(N)$.
In practice $B$ is chosen to fill a GPU efficiently — mini-batch isn't a compromise anyone derives
from first principles so much as the version that happens to match modern hardware.

### Convergence rate and learning rate

For a strongly convex, smooth loss (formalized in `04.03`), batch gradient descent with a
constant learning rate $\eta$ converges **linearly**: the optimality gap $L(\mathbf{w}_k)-L^*$
shrinks by a roughly constant *factor* $\rho<1$ every step, so $L(\mathbf{w}_k)-L^*\approx
\rho^k(L(\mathbf{w}_0)-L^*)$ — a straight line on a log-scale plot, verified numerically below.
SGD cannot match this with a constant $\eta$: because each step's direction is noisy, the
iterates keep bouncing around the minimum rather than settling into it, and convergence in
expectation is only $O(1/k)$ unless $\eta$ is decreased over the run (previewed here, formalized
in `04.06`'s learning rate schedules). The learning rate $\eta$ itself controls a real tradeoff
independent of variant: too small wastes steps, too large overshoots the bowl and can diverge —
the largest stable $\eta$ is set by the curvature of $L$ (its Hessian's largest eigenvalue,
`02.06`), which is exactly why ill-conditioned losses (elongated, elliptical bowls) are painful:
the same $\eta$ that's stable along the shallow direction is too large along the steep one.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Overlaying batch, mini-batch, and SGD trajectories on the same elliptical loss contour, from the
same start, with the same step budget, turns "SGD is noisier" from a claim into something you
can trace with your eyes.

In [ ]:
# Linear regression in 2D weight space with correlated features (an elongated, ill-conditioned
# bowl) -- correlated inputs make the contours elliptical, which is what makes batch vs SGD
# paths visually different rather than just "a straight line either way".
rng = np.random.default_rng(0)
N = 200
true_w = np.array([2.0, -1.0])
cov = np.array([[1.0, 0.85], [0.85, 1.0]])
X_data = rng.multivariate_normal([0, 0], cov, size=N)
y_data = X_data @ true_w + rng.normal(scale=0.3, size=N)

def loss_fn(w, X=X_data, y=y_data):
    return np.mean((X @ w - y) ** 2)

def grad_fn(w, X, y):
    return 2 * X.T @ (X @ w - y) / len(y)

w_star = np.linalg.solve(X_data.T @ X_data, X_data.T @ y_data)  # exact OLS minimizer, for reference

def batch_gd(w0, lr, steps):
    w = w0.copy(); traj = [w.copy()]
    for _ in range(steps):
        w = w - lr * grad_fn(w, X_data, y_data)
        traj.append(w.copy())
    return np.array(traj)

def sgd(w0, lr, steps, rng):
    w = w0.copy(); traj = [w.copy()]
    for _ in range(steps):
        i = rng.integers(0, N)
        grad = 2 * X_data[i] * (X_data[i] @ w - y_data[i])
        w = w - lr * grad
        traj.append(w.copy())
    return np.array(traj)

def minibatch_gd(w0, lr, steps, batch_size, rng):
    w = w0.copy(); traj = [w.copy()]
    for _ in range(steps):
        idx = rng.choice(N, size=batch_size, replace=False)
        w = w - lr * grad_fn(w, X_data[idx], y_data[idx])
        traj.append(w.copy())
    return np.array(traj)

w0 = np.array([-1.2, 1.6])
lr, steps = 0.15, 150
rng2 = np.random.default_rng(1)
traj_batch = batch_gd(w0, lr, steps)
traj_sgd = sgd(w0, lr * 0.3, steps, rng2)          # smaller lr: single-example gradients are noisier
traj_mini = minibatch_gd(w0, lr, steps, 16, rng2)

w1_grid, w2_grid = np.linspace(-2, 3, 150), np.linspace(-1.5, 2, 150)
W1, W2 = np.meshgrid(w1_grid, w2_grid)
Z = np.array([[loss_fn(np.array([a, b])) for a in w1_grid] for b in w2_grid])

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.contour(W1, W2, Z, levels=25, cmap="viridis", alpha=0.6)
ax.plot(*traj_batch.T, "-", color="#4C72B0", lw=2, label=f"batch (all {N}/step)")
ax.plot(*traj_mini.T, "-", color="#DD8452", lw=1.3, alpha=0.85, label="mini-batch (16/step)")
ax.plot(*traj_sgd.T, "-", color="#C44E52", lw=0.9, alpha=0.7, label="SGD (1/step)")
ax.plot(*w0, "ko", markersize=9, label="start")
ax.plot(*w_star, "k*", markersize=16, label="true optimum")
ax.set_xlabel("w1"); ax.set_ylabel("w2")
ax.set_title(f"Same start, same {steps}-step budget, three ways of estimating the gradient")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

def path_stats(traj):
    step_lens = np.linalg.norm(np.diff(traj, axis=0), axis=1)
    return step_lens.sum() / np.linalg.norm(traj[-1] - traj[0])  # "wander ratio": 1.0 = perfectly straight

for name, traj in [("batch", traj_batch), ("mini-batch", traj_mini), ("SGD", traj_sgd)]:
    print(f"{name:10s} final loss={loss_fn(traj[-1]):.4f}  wander ratio={path_stats(traj):.2f}")
print(f"loss at the true optimum: {loss_fn(w_star):.4f}")

## 🐍 Implementation from Scratch

One `gradient_descent` function, parameterized only by `batch_size`, produces all three
variants. We use it to verify the linear-convergence claim for batch GD (a straight line on a
log-scale gap plot) and the noise-floor behavior of constant-learning-rate SGD, then cross-check
the batch case against `torch.optim.SGD`.

In [ ]:
import torch

def gradient_descent(w0, lr, steps, batch_size, rng):
    """One function, three algorithms: batch_size=N is batch GD, batch_size=1 is SGD,
    anything in between is mini-batch -- they differ only in how much data estimates the gradient."""
    w = w0.copy()
    traj = [w.copy()]
    for _ in range(steps):
        idx = np.arange(N) if batch_size >= N else rng.choice(N, size=batch_size, replace=False)
        w = w - lr * grad_fn(w, X_data[idx], y_data[idx])
        traj.append(w.copy())
    return np.array(traj)

loss_star = loss_fn(w_star)
rng3 = np.random.default_rng(2)
traj_b = gradient_descent(w0, lr, steps, N, rng3)
traj_s = gradient_descent(w0, lr * 0.3, steps, 1, rng3)
traj_m = gradient_descent(w0, lr, steps, 16, rng3)

gap_b = np.array([loss_fn(w) - loss_star for w in traj_b])
gap_s = np.array([loss_fn(w) - loss_star for w in traj_s])
gap_m = np.array([loss_fn(w) - loss_star for w in traj_m])

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(gap_b, color="#4C72B0", lw=2, label="batch (straight line = linear convergence)")
ax.semilogy(gap_m, color="#DD8452", lw=1.3, label="mini-batch")
ax.semilogy(gap_s, color="#C44E52", lw=1, label="SGD (settles onto a noise floor)")
ax.set_xlabel("iteration"); ax.set_ylabel("L(w) - L* (log scale)")
ax.set_title("Optimality gap vs iteration: batch decays geometrically,\nconstant-lr SGD stalls at a noise floor")
ax.legend()
plt.show()

# Check the CONSTANT-ratio signature of linear convergence: gap_{k+1}/gap_k should be ~constant
ratios = gap_b[1:31] / gap_b[0:30]
print(f"Batch GD gap ratio, steps 1-10:  {np.round(ratios[:10], 4)}")
print(f"Batch GD gap ratio, steps 21-30: {np.round(ratios[20:30], 4)}")
print(f"-> ratio settles to a constant ({ratios[-1]:.4f}) -- the defining signature of LINEAR convergence.")
print(f"\nSGD gap over the final 50 steps: min={gap_s[-50:].min():.4f}, max={gap_s[-50:].max():.4f}")
print("-> stuck bouncing in a band, never reaching batch GD's ~1e-6 gap, exactly as the Theory predicts.")

# Cross-check deterministic batch GD against torch.optim.SGD (same data, same lr, same steps)
X_t = torch.tensor(X_data, dtype=torch.float64)
y_t = torch.tensor(y_data, dtype=torch.float64)
w_t = torch.tensor(w0.copy(), dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([w_t], lr=lr)
for _ in range(steps):
    opt.zero_grad()
    torch.mean((X_t @ w_t - y_t) ** 2).backward()
    opt.step()
print(f"\ntorch.optim.SGD (full-batch) final w: {w_t.detach().numpy()}")
print(f"our batch GD final w:                {traj_b[-1]}")
print(f"match: {np.allclose(traj_b[-1], w_t.detach().numpy(), atol=1e-6)}")

## 🤖 Why This Matters for AI

Every training loop you've ever seen — `for batch in dataloader: loss.backward(); optimizer.step()`
— is mini-batch gradient descent from this notebook, and the choice of `batch_size` is one of
the first hyperparameters anyone tunes when training a neural network. Batch GD is never used
directly at scale (a single step over a billion-example dataset is far too slow to iterate on),
but pure SGD (`batch_size=1`) is rarely used either, because modern GPUs process a batch of size
16 nearly as fast as a batch of size 1 — mini-batches trade a little gradient noise for a lot of
parallelism. Below, we train a small `torch.nn` regression network at three batch sizes under
the *same total compute budget* (same number of training examples processed) and see this
tradeoff play out directly in final loss.

In [ ]:
import torch.nn as nn

# Same nonlinear regression flavor as 04.01's AI section, now trained with a real torch.nn
# model and torch.optim.SGD, at three batch sizes under a FIXED total example budget.
torch.manual_seed(0)
N_train = 400
X_train = torch.tensor(rng.uniform(-2, 2, size=(N_train, 2)), dtype=torch.float32)
y_train = torch.tensor(
    np.sin(2 * X_train[:, 0].numpy()) + np.cos(X_train[:, 1].numpy()) + rng.normal(0, 0.05, N_train),
    dtype=torch.float32,
).unsqueeze(1)

def make_model():
    torch.manual_seed(0)  # identical initialization across runs -- isolate the batch-size effect
    return nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 1))

def train_torch(batch_size, n_updates, lr):
    model = make_model()
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(n_updates):
        idx = torch.randint(0, N_train, (batch_size,))
        opt.zero_grad()
        loss_fn(model(X_train[idx]), y_train[idx]).backward()
        opt.step()
    with torch.no_grad():
        return loss_fn(model(X_train), y_train).item()

example_budget = 20000  # every configuration processes exactly this many (batch_size x update) pairs
configs = [(1, 0.01, "SGD"), (16, 0.05, "mini-batch"), (400, 0.3, "batch")]

print(f"{'method':12s} {'batch':>6} {'#updates':>9} {'final loss':>11}")
for batch_size, lr, name in configs:
    n_updates = example_budget // batch_size
    final_loss = train_torch(batch_size, n_updates, lr)
    print(f"{name:12s} {batch_size:6d} {n_updates:9d} {final_loss:11.5f}")

print(f"\nAll three see the same {example_budget} training examples in total. SGD gets 20000 noisy")
print("updates, batch GD gets only 50 precise ones -- with a fixed compute budget, MORE noisy")
print("updates beats FEWER precise ones here, which is exactly why SGD/mini-batch dominate")
print("large-scale training rather than batch GD, despite each individual step being noisier.")

## 🏋️ Exercises

### Warm-up
1. For $X=\begin{pmatrix}1&0\\0&1\end{pmatrix}$, $y=(2,1)$, $w=(0,0)$, and loss
   $L(w)=\frac{1}{2}\|Xw-y\|^2$, compute one batch gradient descent step with $\eta=0.1$ by
   hand. Verify with `gradient_descent(w0, lr=0.1, steps=1, batch_size=2, rng=...)`.

### Practice
2. In the Implementation cell, sweep `lr` for batch GD over `[0.05, 0.1, 0.15, 0.2, 0.25]` and
   plot the final optimality gap (after the same number of steps) against `lr`. Find
   approximately where it stops improving and starts diverging, and connect this to the "largest
   stable $\eta$ is set by curvature" claim in the Theory section.

### Challenge
3. In the AI section, hold `batch_size=16` fixed and sweep `lr` over a fine grid (at least 15
   values) to find the learning rate that minimizes final loss under the same `example_budget`.
   Then explain why this optimum shifts if you double `batch_size` to 32 (hint: how does the
   variance of an averaged gradient estimate scale with batch size?).

---

## 📚 Further Reading
- Bottou, Curtis & Nocedal, ["Optimization Methods for Large-Scale Machine Learning"](https://arxiv.org/abs/1606.04838)
- Goyal et al., ["Accurate, Large Minibatch SGD: Training ImageNet in 1 Hour"](https://arxiv.org/abs/1706.02677)
- Robbins & Monro, ["A Stochastic Approximation Method"](https://projecteuclid.org/euclid.aoms/1177729586) (1951, the original SGD convergence analysis)

---

*Next notebook: [Convexity and Convergence Guarantees](03_convexity_and_convergence.ipynb)*